# Reranker GPU: resume an immutable run
Run A ? G on the same Colab GPU kernel. No BM25/dense/RRF or reporting commands.
Known historical output: QASPER reranking succeeded (1005 queries); evaluation and SciFact remain unconfirmed.
The original outputs are preserved in the timestamped notebook backup, not asserted as current runtime state.
A new experiment requires explicit paths and CREATE_NEW=True; Run All never invents a workspace.

## A. Mount Drive and explicit paths

In [13]:
from pathlib import Path
from google.colab import drive
import datetime, hashlib, json, os, re, shutil, subprocess, sys, tempfile, traceback, uuid, zipfile
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
assert Path('/content/drive/MyDrive').is_dir()
DRIVE_OUTPUT = Path('/content/drive/MyDrive/retrieval_benchmark/reranker_runs')
INPUT_ZIP = Path('/content/drive/MyDrive/retrieval_benchmark/reranker_benchmark_input_v1.zip')
# Explicit existing session. Set None only to discover other configs; never auto-fallback.
CONFIG_SELECTION = '/content/reranker_20261031T164343Z_05af25c1/reranker_run_config.json'
# Known session: /content/reranker_20261031T164343Z_05af25c1/reranker_run_config.json
# If runtime was lost, choose its Drive config above and a verified G archive below.
RESTORE_ZIP = None
RESTORE_SHA256 = None
CREATE_NEW = False
NEW_WORK = None                  # explicit /content/reranker_<chosen-name>
NEW_DRIVE_RUN = None             # explicit directory under DRIVE_OUTPUT
MODEL_SHA = '233902d25c440f23af6f7d6e94d2946bac0bee0a'
SOURCE_COMMIT = '1d3ab1035545a867be1c649be17ad22565303e1f'
SOURCE_URL = 'https://github.com/JustKha05/local-scientific-research-agent.git'

## B. Select/restore workspace, read config, inspect status

In [14]:
def sha(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024*1024), b''): h.update(block)
    return h.hexdigest()
def read(path): return json.loads(Path(path).read_text(encoding='utf-8'))
def write_new(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('x', encoding='utf-8') as f:
        json.dump(value, f, ensure_ascii=False, indent=2, allow_nan=False)
def extract_safe(archive_path, target):
    with zipfile.ZipFile(archive_path) as z:
        for m in z.infolist():
            assert (target / m.filename).resolve().is_relative_to(target.resolve()), m.filename
            assert (m.external_attr >> 16) & 0o170000 != 0o120000, 'ZIP symlink'
        z.extractall(target)
candidates = sorted(set(Path('/content').glob('reranker_*/reranker_run_config.json')) |
                    set(DRIVE_OUTPUT.glob('*/reranker_run_config.json')))
for candidate in candidates: print('CONFIG:', candidate)
if CONFIG_SELECTION is None:
    assert len(candidates) == 1 or (not candidates and CREATE_NEW), (
        'Set CONFIG_SELECTION to one exact config path from the list; no automatic session selection')
    selected = candidates[0] if candidates else None
else:
    selected = Path(CONFIG_SELECTION)
if selected is None:
    assert CREATE_NEW and NEW_WORK and NEW_DRIVE_RUN, 'First run requires explicit paths'
    WORK = Path(NEW_WORK); DRIVE_RUN = Path(NEW_DRIVE_RUN)
    assert WORK.parent == Path('/content') and WORK.name.startswith('reranker_')
    assert DRIVE_RUN.parent == DRIVE_OUTPUT
    assert not WORK.exists() and not DRIVE_RUN.exists(), 'Reuse existing config instead'
    WORK.mkdir(); DRIVE_RUN.mkdir(parents=True)
    extract_safe(INPUT_ZIP, WORK)
    CONFIG_PATH = WORK / 'reranker_run_config.json'
    write_new(CONFIG_PATH, dict(model_id='cross-encoder/ms-marco-MiniLM-L6-v2',
        revision=MODEL_SHA, tokenizer_revision=MODEL_SHA, device='cuda', batch_size=16,
        max_length=512, top_k=10, source_depth=50, reporting_unlocked=False,
        bm25_dir='evaluation/reports/dev_bm25_top50_v1',
        dense_dir='evaluation/reports/dev_dense_top50_v1',
        report_dir='evaluation/reports/dev_cross_encoder_v1',
        input_zip_sha256=sha(INPUT_ZIP), run_id=DRIVE_RUN.name))
    shutil.copy2(CONFIG_PATH, DRIVE_RUN / CONFIG_PATH.name)
else:
    assert selected.is_file(), selected
    if selected.is_relative_to(DRIVE_OUTPUT):
        DRIVE_RUN = selected.parent
        WORK = Path('/content') / ('reranker_' + read(selected)['run_id'])
        if not WORK.exists():
            assert RESTORE_ZIP and RESTORE_SHA256, 'Select a G archive and its SHA256 to restore'
            assert sha(RESTORE_ZIP) == RESTORE_SHA256, 'Restore ZIP hash mismatch'
            # Extract into staging first; failed restore never becomes a workspace.
            stage = Path(tempfile.mkdtemp(prefix='restore_', dir='/content'))
            extract_safe(RESTORE_ZIP, stage)
            assert (stage / 'reranker_run_config.json').read_bytes() == selected.read_bytes()
            stage.rename(WORK)
    else:
        WORK = selected.parent
        DRIVE_RUN = DRIVE_OUTPUT / read(selected)['run_id']
    CONFIG_PATH = WORK / 'reranker_run_config.json'
CONFIG = read(CONFIG_PATH)
assert CONFIG['revision'] == CONFIG['tokenizer_revision'] == MODEL_SHA
assert CONFIG['model_id'] == 'cross-encoder/ms-marco-MiniLM-L6-v2'
assert CONFIG['device'] == 'cuda' and CONFIG['source_depth'] == 50
assert CONFIG.get('reporting_unlocked') is False
assert CONFIG['top_k'] == 10 and CONFIG['max_length'] == 512 and CONFIG['batch_size'] == 16
assert (DRIVE_RUN / CONFIG_PATH.name).read_bytes() == CONFIG_PATH.read_bytes(), 'Drive/local config mismatch'
if INPUT_ZIP.is_file(): assert sha(INPUT_ZIP) == CONFIG['input_zip_sha256']
for key in ('bm25_dir', 'dense_dir', 'report_dir'):
    assert not Path(CONFIG[key]).is_absolute()
    assert (WORK / CONFIG[key]).resolve().is_relative_to(WORK.resolve())
REPORT_DIR = WORK / CONFIG['report_dir']
CONFIG_HASH = sha(CONFIG_PATH)
LOG_DIR = WORK / 'resume_logs'; LOG_DIR.mkdir(exist_ok=True)
PROVENANCE = WORK / 'resume_provenance'; PROVENANCE.mkdir(exist_ok=True)
def token(): return datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')+'_'+uuid.uuid4().hex[:8]
def run(command, label):
    log = LOG_DIR / (label + '_' + token() + '.log')
    print('COMMAND:', command, 'cwd:', WORK, 'log:', log, flush=True)
    with log.open('x', encoding='utf-8') as f:
        with subprocess.Popen([str(x) for x in command], cwd=WORK,
                stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                encoding='utf-8', errors='replace', bufsize=1,
                env={**os.environ, 'PYTHONUNBUFFERED':'1'}) as proc:
            for line in proc.stdout:
                print(line, end='', flush=True); f.write(line); f.flush()
            rc = proc.wait()
        print('Exit code:', rc); f.write(f'\nExit code: {rc}\n')
    if rc: raise subprocess.CalledProcessError(rc, command)  # original child traceback already streamed/logged
    return log
for d in ('qasper','scifact'):
    print(d, {s:(REPORT_DIR/f'{d}_{s}.json').is_file()
              for s in ('retrieval','metrics_development','diagnosis')})
print('WORK:', WORK, '\nCONFIG SHA256:', CONFIG_HASH, '\nDrive:', DRIVE_RUN)

CONFIG: /content/drive/MyDrive/retrieval_benchmark/reranker_runs/20261030T165049Z_5bf40cce/reranker_run_config.json
CONFIG: /content/drive/MyDrive/retrieval_benchmark/reranker_runs/20261031T164343Z_05af25c1/reranker_run_config.json
CONFIG: /content/reranker_20261031T164343Z_05af25c1/reranker_run_config.json
qasper {'retrieval': False, 'metrics_development': False, 'diagnosis': False}
scifact {'retrieval': False, 'metrics_development': False, 'diagnosis': False}
WORK: /content/reranker_20261031T164343Z_05af25c1 
CONFIG SHA256: b162a087f864e03897c127a069b8e98551b3cc1a1d3e63803f17e2fa40ea31c7 
Drive: /content/drive/MyDrive/retrieval_benchmark/reranker_runs/20261031T164343Z_05af25c1


## C. Synchronize source from an exact Git commit
Preserve original source bytes before synchronization. A legacy report must match these bytes or the pinned source; unknown script hashes stop the run. This records provenance, without claiming that old source came from the new commit.

In [15]:
assert re.fullmatch('[0-9a-f]{40}', SOURCE_COMMIT)
# Preserve all original Python source, including the prior in-kernel loader fix.
for folder in ('scripts', 'app'):
    for path in (WORK / folder).glob('*.py'):
        dest = PROVENANCE / 'source_blobs' / sha(path)
        if not dest.exists():
            dest.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(path, dest)
source_dir = WORK / ('source_' + SOURCE_COMMIT)
if not source_dir.exists():
    source_dir.mkdir()
    run(['git','init',source_dir], 'git_init')
    run(['git','-C',source_dir,'remote','add','origin',SOURCE_URL], 'git_remote')
    run(['git','-C',source_dir,'fetch','--depth','1','origin',SOURCE_COMMIT], 'git_fetch')
    run(['git','-C',source_dir,'checkout','--detach',SOURCE_COMMIT], 'git_checkout')
# No branch, tag or main resolution. rev-parse stdout is logged as evidence.
head_log = run(['git','-C',source_dir,'rev-parse','HEAD'], 'git_head')
assert head_log.read_text().splitlines()[0] == SOURCE_COMMIT
run(['git','-C',source_dir,'diff','--exit-code','HEAD','--','scripts','app','requirements.txt'], 'git_clean')
assert 'snapshot_download' in (source_dir/'scripts/rerank_retrieval.py').read_text()
source_hashes = {}
for folder in ('scripts','app'):
    expected = {p.name for p in (source_dir/folder).glob('*.py')}
    assert {p.name for p in (WORK/folder).glob('*.py')} <= expected, 'Unexpected Python source; inspect manually'
    for src in (source_dir/folder).glob('*.py'):
        dest = WORK / folder / src.name
        dest.parent.mkdir(exist_ok=True); shutil.copy2(src, dest)
        source_hashes[dest.relative_to(WORK).as_posix()] = sha(dest)
shutil.copy2(source_dir/'requirements.txt', WORK/'requirements.txt')
source_hashes['requirements.txt'] = sha(WORK/'requirements.txt')
SOURCE_RECORD = dict(commit=SOURCE_COMMIT, repository=SOURCE_URL, files=source_hashes,
                     config_sha256=CONFIG_HASH)
record = PROVENANCE / ('source_'+SOURCE_COMMIT+'.json')
if record.exists(): assert read(record) == SOURCE_RECORD
else: write_new(record, SOURCE_RECORD)
# Reload validators if this kernel previously imported an older workspace source.
os.chdir(WORK)
sys.path.insert(0, str(WORK))
for module in list(sys.modules):
    if module == 'scripts' or module.startswith('scripts.') or module == 'app' or module.startswith('app.'):
        del sys.modules[module]
print('Pinned source:', SOURCE_COMMIT)

COMMAND: ['git', 'init', PosixPath('/content/reranker_20261031T164343Z_05af25c1/source_1d3ab1035545a867be1c649be17ad22565303e1f')] cwd: /content/reranker_20261031T164343Z_05af25c1 log: /content/reranker_20261031T164343Z_05af25c1/resume_logs/git_init_20261031T165945973706Z_1ffbb774.log
hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /content/reranker_20261031T164343Z_05af25c1/source_1d3ab1035545a867be1c649be17ad22565303e1f/.git/
Exit code: 0
COMMAND: ['git', '-C', PosixPath('/content/reranker_20261031T164343Z_05af25c1/sour

## D. Dependencies, CUDA and fixture tests (no benchmark inference in tests)

In [16]:
run([sys.executable,'-u','-m','pip','install','-r','requirements.txt'], 'dependencies')
run([sys.executable,'-u','-c',
     "import torch; print('torch',torch.__version__,'CUDA',torch.version.cuda); "
     "assert torch.cuda.is_available(), 'Enable Colab GPU; no CPU fallback'; "
     "print(torch.cuda.get_device_name(0))"], 'cuda')
run([sys.executable,'-u','-m','unittest','scripts.test_rerank_retrieval',
     'scripts.test_fuse_retrieval_rrf','-v'], 'fixture_tests')

COMMAND: ['/usr/bin/python3', '-u', '-m', 'pip', 'install', '-r', 'requirements.txt'] cwd: /content/reranker_20261031T164343Z_05af25c1 log: /content/reranker_20261031T164343Z_05af25c1/resume_logs/dependencies_20261031T165946848514Z_50ea5dcb.log
Exit code: 0
COMMAND: ['/usr/bin/python3', '-u', '-c', "import torch; print('torch',torch.__version__,'CUDA',torch.version.cuda); assert torch.cuda.is_available(), 'Enable Colab GPU; no CPU fallback'; print(torch.cuda.get_device_name(0))"] cwd: /content/reranker_20261031T164343Z_05af25c1 log: /content/reranker_20261031T164343Z_05af25c1/resume_logs/cuda_20261031T165948302605Z_31de6206.log
torch 2.11.0+cu130 CUDA 13.0
Tesla T4
Exit code: 0
COMMAND: ['/usr/bin/python3', '-u', '-m', 'unittest', 'scripts.test_rerank_retrieval', 'scripts.test_fuse_retrieval_rrf', '-v'] cwd: /content/reranker_20261031T164343Z_05af25c1 log: /content/reranker_20261031T164343Z_05af25c1/resume_logs/fixture_tests_20261031T165950190523Z_932783ea.log
test_conflict_and_source_

PosixPath('/content/reranker_20261031T164343Z_05af25c1/resume_logs/fixture_tests_20261031T165950190523Z_932783ea.log')

## E. Validate files and rerank only missing retrieval reports

In [17]:
from scripts.benchmark_common import load_inputs, index, validate_manifest
from scripts.evaluate_retrieval_benchmark import validate_report, score_query
from scripts.fuse_retrieval_rrf import validate_sources
from scripts.rerank_retrieval import candidate_union

def verify_source():
    assert sha(CONFIG_PATH) == CONFIG_HASH, 'Config changed'
    assert (DRIVE_RUN/CONFIG_PATH.name).read_bytes() == CONFIG_PATH.read_bytes()
    for rel, digest in SOURCE_RECORD['files'].items():
        assert sha(WORK/rel) == digest, f'Source changed: {rel}'
def validate_dataset(d):
    verify_source()
    qs, ds, hashes = load_inputs(WORK/'data/processed', d, 'dev')
    queries, documents = index(qs,'query_id'), index(ds,'evidence_id')
    manifest_path = WORK/f'configs/dev_{d}_seed42.json'
    assignments = validate_manifest(read(manifest_path), qs, d, hashes['queries_sha256'])
    paths = [WORK/CONFIG[k]/f'{d}_retrieval.json' for k in ('bm25_dir','dense_dir')]
    sources = [read(p) for p in paths]
    left, right = validate_sources(*sources,d)
    for source in sources:
        validate_report(source,d,'dev',queries,documents,hashes)
        assert source['manifest_sha256'] == sha(manifest_path)
    unions = {qid:candidate_union(row,right[qid]) for qid,row in left.items()}
    path = REPORT_DIR/f'{d}_retrieval.json'
    if path.exists():
        report = read(path)
        rows = validate_report(report,d,'dev',queries,documents,hashes)
        assert report['manifest_sha256'] == sha(manifest_path)
        for key in ('model_id','revision','tokenizer_revision','device','batch_size','max_length','top_k','source_depth'):
            assert report['config'][key] == CONFIG[key], f'{d}: config mismatch {key}'
        assert report['config']['retriever'] == 'cross_encoder'
        assert report['config']['dtype'] == 'float32'
        for field, rel in [('script_sha256','scripts/rerank_retrieval.py'),
                           ('validation_script_sha256','scripts/fuse_retrieval_rrf.py')]:
            digest = report[field]
            blob = PROVENANCE/'source_blobs'/digest
            assert digest == sha(WORK/rel) or (blob.is_file() and sha(blob)==digest), (
                f'{d}: unknown source hash {field}; recover exact original source before reuse')
        for method,p in zip(('bm25','dense'),paths):
            assert report['rerank_sources'][method]['sha256'] == sha(p), f'{d}: source report hash mismatch'
        for qid,row in rows.items():
            union_ids = {h['evidence_id'] for h in unions[qid]}
            assert row['rerank_candidate_count'] == len(union_ids)
            assert len(row['hits']) == min(CONFIG['top_k'],len(union_ids))
            assert {h['evidence_id'] for h in row['hits']} <= union_ids
        print(d, 'VALID retrieval;', len(rows), 'queries; SHA256:', sha(path))
    else:
        assert not any((REPORT_DIR/f'{d}_{s}.json').exists() for s in ('metrics_development','diagnosis')), 'Orphan evaluation'
    return queries,documents,hashes,assignments

def save_failure(stage):
    path = LOG_DIR / ('FAILED_'+stage+'_'+token()+'.json')
    write_new(path, {'stage':stage,'status':'FAILED','traceback':traceback.format_exc()})
    # Failure evidence on Drive immediately; G can archive all partial artifacts later.
    shutil.copy2(path, DRIVE_RUN/path.name)
try:
    # Validate BOTH datasets before launching any expensive work.
    for d in ('qasper','scifact'): validate_dataset(d)
    for d in ('qasper','scifact'):
        if not (REPORT_DIR/f'{d}_retrieval.json').exists():
            run([sys.executable,'-u','-m','scripts.rerank_retrieval','--dataset',d,
                 '--bm25-dir',CONFIG['bm25_dir'],'--dense-dir',CONFIG['dense_dir'],
                 '--report-dir',CONFIG['report_dir'],'--model-id',CONFIG['model_id'],
                 '--revision',CONFIG['revision'],'--device','cuda',
                 '--batch-size',str(CONFIG['batch_size']),'--max-length',str(CONFIG['max_length']),
                 '--top-k',str(CONFIG['top_k'])], d+'_rerank')
        validate_dataset(d)
except BaseException:
    save_failure('rerank')
    raise

COMMAND: ['/usr/bin/python3', '-u', '-m', 'scripts.rerank_retrieval', '--dataset', 'qasper', '--bm25-dir', 'evaluation/reports/dev_bm25_top50_v1', '--dense-dir', 'evaluation/reports/dev_dense_top50_v1', '--report-dir', 'evaluation/reports/dev_cross_encoder_v1', '--model-id', 'cross-encoder/ms-marco-MiniLM-L6-v2', '--revision', '233902d25c440f23af6f7d6e94d2946bac0bee0a', '--device', 'cuda', '--batch-size', '16', '--max-length', '512', '--top-k', '10'] cwd: /content/reranker_20261031T164343Z_05af25c1 log: /content/reranker_20261031T164343Z_05af25c1/resume_logs/qasper_rerank_20261031T170000012672Z_250ae62c.log

Fetching 23 files:   0%|          | 0/23 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

Fetching 23 files: 100%|██████████| 23/23 [00:05<00:00,  4.17it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 4633.47it/s]
Saved evaluation/reports/dev_cross_encoder_v1/qaspe

## F. Diagnosis/calibration evaluation and verified status
Existing metrics must bind to the current retrieval, data, gold and manifest. Partial evaluation is rebuilt in staging; matching existing files are kept and only missing files are published. Conflicts stop without overwriting.

In [18]:
def check_evaluation(d):
    queries,docs,hashes,assignments = validate_dataset(d)
    metric = REPORT_DIR/f'{d}_metrics_development.json'
    diagnosis = REPORT_DIR/f'{d}_diagnosis.json'
    if metric.exists():
        m = read(metric)
        assert m['dataset']==d and m['split']=='dev' and m['retriever']=='cross_encoder'
        assert m['reporting_unlocked'] is False and set(m['partitions'])=={'diagnosis','calibration'}
        assert m['retrieval_sha256']==sha(REPORT_DIR/f'{d}_retrieval.json')
        assert m['gold_sha256']==sha(WORK/f'data/processed/{d}/dev/gold.jsonl')
        assert m['source_hashes']==hashes
        assert m['manifest_sha256']==sha(WORK/f'configs/dev_{d}_seed42.json')
    # Recompute expected evaluation in memory, using the pinned evaluator and current gold.
    # No subprocess and no writes for complete reports; validates metric/diagnosis content too.
    from scripts import evaluate_retrieval_benchmark as evaluator
    from types import SimpleNamespace
    expected = {}
    original = evaluator.write_new
    def capture(path,value): expected[Path(path).name] = value
    evaluator.write_new = capture
    try:
        import contextlib, io
        with contextlib.redirect_stdout(io.StringIO()):
            evaluator.evaluate(SimpleNamespace(dataset=d,split='dev',processed_dir=WORK/'data/processed',
                report_dir=REPORT_DIR,query_manifest=WORK/f'configs/dev_{d}_seed42.json',include_reporting=False))
    finally: evaluator.write_new = original
    for name,value in expected.items():
        path = REPORT_DIR/name
        if path.exists(): assert read(path)==value, f'Corrupt/stale evaluation: {path}'
    return expected
STATUS = {}
try:
    for d in ('qasper','scifact'):
        expected = check_evaluation(d)
        missing = [name for name in expected if not (REPORT_DIR/name).exists()]
        if missing:
            stage = Path(tempfile.mkdtemp(prefix=d+'_eval_',dir=WORK))
            shutil.copy2(REPORT_DIR/f'{d}_retrieval.json',stage/f'{d}_retrieval.json')
            run([sys.executable,'-u','-m','scripts.evaluate_retrieval_benchmark',
                 '--dataset',d,'--split','dev','--processed-dir',WORK/'data/processed',
                 '--query-manifest',WORK/f'configs/dev_{d}_seed42.json','--report-dir',stage], d+'_evaluate')
            for name,value in expected.items():
                assert read(stage/name)==value
                target = REPORT_DIR/name
                if target.exists(): assert read(target)==value
                else:
                    with target.open('xb') as f: f.write((stage/name).read_bytes())
        check_evaluation(d)
        STATUS[d] = 'VERIFIED_COMPLETE'
    files = {p.relative_to(WORK).as_posix():sha(p) for p in sorted(REPORT_DIR.glob('*.json'))}
    write_new(PROVENANCE/('verified_'+token()+'.json'),dict(status=STATUS,config_sha256=CONFIG_HASH,
              source=SOURCE_RECORD,files=files))
except BaseException:
    save_failure('evaluation')
    raise
finally:
    print('dataset | retrieval | evaluation | status')
    for d in ('qasper','scifact'):
        print(d, '|', (REPORT_DIR/f'{d}_retrieval.json').exists(), '|',
              (REPORT_DIR/f'{d}_metrics_development.json').exists(), '|', STATUS.get(d,'NOT_VERIFIED'))

qasper VALID retrieval; 1005 queries; SHA256: bd9211644c4a193b79cbb64852ad7efb0669b3723137dd407c021372ff0b415b
COMMAND: ['/usr/bin/python3', '-u', '-m', 'scripts.evaluate_retrieval_benchmark', '--dataset', 'qasper', '--split', 'dev', '--processed-dir', PosixPath('/content/reranker_20261031T164343Z_05af25c1/data/processed'), '--query-manifest', PosixPath('/content/reranker_20261031T164343Z_05af25c1/configs/dev_qasper_seed42.json'), '--report-dir', PosixPath('/content/reranker_20261031T164343Z_05af25c1/qasper_eval_7mvwmye2')] cwd: /content/reranker_20261031T164343Z_05af25c1 log: /content/reranker_20261031T164343Z_05af25c1/resume_logs/qasper_evaluate_20261031T170308261070Z_b49a4aef.log
{
  "diagnosis": {
    "selected_queries": 416,
    "groups": 113,
    "scored_queries": 341,
    "skipped_count": 75,
    "skipped_reasons": {
      "no_eligible_text_annotation": 75
    },
    "statuses": {
      "miss": 57,
      "complete": 204,
      "complete_alternative": 29,
      "partial": 51
    

## G. Versioned Drive backup (run also after a failed E/F)

In [19]:
# Re-validate before labelling a backup complete; never rely on STATUS from a prior cell run.
backup_status = 'FAILED_OR_INCOMPLETE'
try:
    for d in ('qasper','scifact'):
        expected = check_evaluation(d)
        assert all((REPORT_DIR/name).is_file() for name in expected)
    backup_status = 'VERIFIED_COMPLETE'
except BaseException:
    save_failure('backup_validation')
    traceback.print_exc()
# Include input data and source reports so a fresh runtime can resume without a second ZIP.
paths = [CONFIG_PATH,WORK/'requirements.txt']
for folder in ('scripts','app','configs','data/processed','resume_logs','resume_provenance'):
    paths += [p for p in (WORK/folder).rglob('*') if p.is_file() and '__pycache__' not in p.parts]
for directory in (WORK/CONFIG['bm25_dir'],WORK/CONFIG['dense_dir'],REPORT_DIR):
    paths += list(directory.glob('*.json'))
paths = sorted(set(paths))
manifest = dict(status=backup_status,source_commit=SOURCE_COMMIT,config_sha256=sha(CONFIG_PATH),
                files={p.relative_to(WORK).as_posix():sha(p) for p in paths})
archive = DRIVE_RUN / ('artifacts_'+backup_status+'_'+token()+'.zip')
try:
    with zipfile.ZipFile(archive,'x',compression=zipfile.ZIP_DEFLATED) as z:
        for p in paths: z.write(p,p.relative_to(WORK).as_posix())
        z.writestr('backup_manifest.json',json.dumps(manifest,indent=2))
    with zipfile.ZipFile(archive) as z:
        assert z.testzip() is None
        for rel,digest in manifest['files'].items():
            assert hashlib.sha256(z.read(rel)).hexdigest()==digest
    write_new(archive.with_suffix('.sha256.json'),dict(file=archive.name,sha256=sha(archive),status=backup_status))
except BaseException:
    save_failure('archive_write')
    raise
print('Backup status:', backup_status, '\nArchive:',archive,'\nSHA256:',sha(archive))
if backup_status != 'VERIFIED_COMPLETE':
    raise RuntimeError('Partial/failure backup saved; benchmark is NOT complete. Inspect FAILED logs.')

qasper VALID retrieval; 1005 queries; SHA256: bd9211644c4a193b79cbb64852ad7efb0669b3723137dd407c021372ff0b415b
scifact VALID retrieval; 300 queries; SHA256: ca8b79114781108d787836b29e78c8e76596134ba257ec193d503d2130e48bfd
Backup status: VERIFIED_COMPLETE 
Archive: /content/drive/MyDrive/retrieval_benchmark/reranker_runs/20261031T164343Z_05af25c1/artifacts_VERIFIED_COMPLETE_20261031T170355811993Z_1bcc0097.zip 
SHA256: a2aa11be3c93861739d79ebceb3219b3dc50d8b95d1b06ac93aaeb6011051bc0
